# Optimizing low-autocorrelation binary sequences

Generate the full original objective, select the circuit by measured CX, then train and sample in FlagQuantum.

## Separate explicit Circuit search from synth optimization

This workflow constructs three exact 13-bit Circuit candidates for the same
162-term cost and three layers. quantum.py explicitly orders parity terms,
estimates actual CX counts and selects the least-CX candidate. This is an
application-owned Circuit implementation and bounded host search, not a qfunc
or proof that synth automatically discovers these schedules. All candidates
are checked against independent complete-state references, including identity
phase corrections.

Try inspecting another Schedule and compare full-state error before comparing
CX count or depth. Changing an angle binding must preserve parameter names and
low-bit-to-wire mapping. A lower CX count is specific to the declared CX/U3
basis; it is not a hardware-runtime or optimization-success guarantee.


Generate the original 13-spin autocorrelation cost independently.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.low_autocorrelation_binary_sequences_problem.evidence_scaling_labs.overview import (
    checks,
    model,
    plots,
    quantum,
    workflow,
)
from examples.synthesis.applications.optimization.low_autocorrelation_binary_sequences_problem.evidence_scaling_labs.overview.main import (
    report,
)

terms = model.cost_terms()


Keep all 13 spins and both constant and variable cost terms.

In [ ]:
assert model.WIDTH == 13 and len(terms) == 162
print("Source cost for the zero word:", model.source_cost(0))


Construct the same three alternating cost and mixer layers.

In [ ]:
assert model.LAYERS == 3


Generate and measure candidate parity orders in a common CX/U3 basis.

In [ ]:
candidates, selected = quantum.choose()
assert selected.cx == min(candidate.cx for candidate in candidates)


Check the complete compiled states at two nonzero sets of angles.

In [ ]:
errors = checks.verify(candidates)
print("Maximum complete-state error:", max(errors))


Run the original 60-query optimization budget using the measured CX winner.

In [ ]:
result = workflow.run()
assert result.selected.schedule is selected.schedule
report(result, errors)


Plot actual mean costs from the executed objective calls.

In [ ]:
figures = Path("notebook-figures")
plots.save(result, figures)
display(Image(filename=str(figures / "cost-convergence.png")))


Inspect five of the lowest-cost outcomes among the actual 1,000 draws.

In [ ]:
print("First five observed rows:", result.samples[:5])


Plot the measured cost distribution from those draws.

In [ ]:
display(Image(filename=str(figures / "observed-costs.png")))
